In [ ]:
!pip install yt-dlp openai-whisper transformers torch

import yt_dlp
import whisper
import torch
import os
import glob
from google.colab import files
from transformers import pipeline

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=TLKxdTmk-zc"
def download_audio(url):
    print("--- Starting Download ---")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': 'audio_file.%(ext)s',
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return "audio_file.mp3"

def transcribe_audio(file_path):
    model = whisper.load_model("base")

    result = model.transcribe(file_path)
    text = result["text"]

    with open("transcript.txt", "w", encoding="utf-8") as f:
        f.write(text)
    return text

def summarize_transcript(text):
    print("--- Loading BART Summarizer (Manual Load) ---")

    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    import torch

    model_name = "facebook/bart-large-cnn"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    words = text.split()
    chunks = [" ".join(words[i : i + 500]) for i in range(0, len(words), 500)]

    full_summary = []

    for i, chunk in enumerate(chunks):
        if len(chunk.split()) < 30: continue

        inputs = tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True).to(device)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=150,
            min_length=40,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

        result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        full_summary.append(result)

    return " ".join(full_summary)

try:
    audio_file = download_audio(VIDEO_URL)

    transcript = transcribe_audio(audio_file)
    print(transcript[:200] + "...")

    final_summary = summarize_transcript(transcript)

    print("\n" + "="*40)
    print("FINAL SUMMARY")
    print("="*40)
    print(final_summary)

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
! pip install deep_translator gtts

In [ ]:


from deep_translator import GoogleTranslator
from gtts import gTTS
from IPython.display import Audio, display

def translate_and_read(english_text, target_lang='hi'):
    # Step A: Translate the English summary to the target language
    print(f"Translating summary to {target_lang}...")
    translated_text = GoogleTranslator(source='en', target=target_lang).translate(english_text)

    print(f"Translated Text Preview: {translated_text[:100]}...")

    print("Generating audio...")
    tts = gTTS(text=translated_text, lang=target_lang)
    tts.save("multilingual_summary.mp3")

    display(Audio("multilingual_summary.mp3", autoplay=False))

print("In which language would you like to hear the summary?")
print("1.  English (en)")
print("2.  Kannada (kn)")
print("3.  Hindi (hi)")
print("4.  Tamil (ta)")
print("5.  Telugu (te)")
print("6.  Malayalam (ml)")
print("7.  Marathi (mr)")
print("8.  Bengali (bn)")
print("9.  Gujarati (gu)")
print("10. Spanish (es)")
print("11. French (fr)")
print("12. Japanese (ja)")
target=input("")
translate_and_read(final_summary, target_lang=target)